In [ ]:
%run ../../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [12]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [6]:
@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"


In [7]:

sql_query.invoke("SELECT * FROM Artist LIMIT 5")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]"

In [4]:
weak_agent = create_agent(
    model=create_noreason_llm(model="databricks-gemma-3-12b"),
    tools=[sql_query]
)

messages = [
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

weak_response = weak_agent.invoke({"messages": messages})
pp(weak_response['messages'][-1].content)

"I am unable to answer the question because the table 'artists' does not exist in the database."


In [5]:
for msg in weak_response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Who is the most popular artist beginning with 'S' in this database?
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_6f3894fe-9dff-4cf4-8671-1e999ca0b326)
 Call ID: call_6f3894fe-9dff-4cf4-8671-1e999ca0b326
  Args:
    query: SELECT artist FROM artists WHERE artist LIKE 'S%' ORDER BY popularity DESC LIMIT 1
================================= Tool Message =================================
Name: sql_query

Error: (sqlite3.OperationalError) no such table: artists
[SQL: SELECT artist FROM artists WHERE artist LIKE 'S%' ORDER BY popularity DESC LIMIT 1]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================

I am unable to answer the question because the table 'artists' does not exist in the database.


In [8]:
from langchain.messages import HumanMessage, SystemMessage

schema_hint = """
Use only existing tables/columns from this SQLite schema:
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.Quantity).
"""

messages = [
    SystemMessage(content=schema_hint),
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response = weak_agent.invoke({"messages": messages})
pp(response['messages'][-1].content)

"The most popular artist beginning with 'S' is Smashing Pumpkins."


In [9]:
for msg in response['messages']:
    msg.pretty_print()

================================ System Message ================================


Use only existing tables/columns from this SQLite schema:
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.Quantity).

================================ Human Message =================================

Who is the most popular artist beginning with 'S' in this database?
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_e1fa6058-e578-45de-bfa0-0e2198bfb0f2)
 Call ID: call_e1fa6058-e578-45de-bfa0-0e2198bfb0f2
  Args:
    query: SELECT A.Name FROM Artist AS A JOIN Album AS AL ON A.ArtistId = AL.ArtistId JOIN Track AS T ON AL.AlbumId = T.AlbumId JOIN Invoic

## Dynamic schema in system message
This section demonstrates how to extract the schema at runtime and append it to the system message before invoking the agent.  But first, some python pre-requisite(s)...

In [14]:
# `ast` stands for Abstract Syntax Trees.  It is used to safely parse
# string-ified Python literals (e.g., arrays/list/tuple results) into actual Python objects.
# But it does not evaluate any python code in the string!

import ast

ast.literal_eval("[1, 2, 3]")
# → [1, 2, 3]

ast.literal_eval("{'a': 1, 'b': 2}")
# → {'a': 1, 'b': 2}

ast.literal_eval("(True, None, 3.14)")
# → (True, None, 3.14)


(True, None, 3.14)

In [15]:
def get_sqlite_schema_overview(database) -> str:
    """Build table(column1, column2, ...) lines from SQLite metadata."""
    
    # Query SQLite system metadata to get all non-internal table names, then safely
    # parse the SQLDatabase string output into Python rows for schema introspection.
    table_rows = ast.literal_eval(database.run(
        """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
            ORDER BY name;
            """
        )
    )

    schema_lines = []
    for row in table_rows:
        table_name = row[0]
        
        # PRAGMA keyword tells SQLite to return metadata about the columns of the specified table.
        column_rows = ast.literal_eval(database.run(f"PRAGMA table_info('{table_name}')"))
        column_names = [col[1] for col in column_rows]
        if column_names:
            schema_lines.append(f"{table_name}({', '.join(column_names)})")

    return "\n".join(schema_lines)

In [16]:

schema_overview_dynamic = get_sqlite_schema_overview(db)
print(schema_overview_dynamic)

Album(AlbumId, Title, ArtistId)
Artist(ArtistId, Name)
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email)
Genre(GenreId, Name)
Invoice(InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)


# SQLite PRAGMA Statement Explanation

The `PRAGMA table_info()` statement above is a SQLite metadata query that returns information about the columns in a specified table.

In the response above, you see results like:
```
[(0, 'TrackId', 'INTEGER', 1, None, 1), (1, 'Name', 'NVARCHAR(200)', 0, None, 0), ...]
```

Each tuple represents one column with the following structure:

| Index | Field | Description |
|-------|-------|-------------|
| 0 | **cid** | Column ID (0-indexed position in table) |
| 1 | **name** | Column name (e.g., 'TrackId', 'Name') |
| 2 | **type** | Data type (e.g., 'INTEGER', 'NVARCHAR(200)') |
| 3 | **notnull** | 1 if NOT NULL constraint exists, 0 otherwise |
| 4 | **dflt_value** | Default value for the column (None if not set) |
| 5 | **pk** | 1 if column is part of primary key, 0 otherwise |

**Example interpretation:**
- `(0, 'TrackId', 'INTEGER', 1, None, 1)` → TrackId is an INTEGER, NOT NULL, and the primary key
- `(1, 'Name', 'NVARCHAR(200)', 0, None, 0)` → Name is NVARCHAR(200), nullable, not a primary key

## Helping our weak_agent to work better ...

In [17]:
dynamic_schema_hint = f"""
Use only existing tables/columns from this SQLite schema:
{schema_overview_dynamic}

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.Quantity).
"""

dynamic_messages = [
    SystemMessage(content=dynamic_schema_hint),
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response_dynamic = weak_agent.invoke({"messages": dynamic_messages})
pp(response_dynamic['messages'][-1].content)

"The most popular artist beginning with 'S' is Smashing Pumpkins."


In [18]:
for msg in response_dynamic['messages']:
    msg.pretty_print()

================================ System Message ================================


Use only existing tables/columns from this SQLite schema:
Album(AlbumId, Title, ArtistId)
Artist(ArtistId, Name)
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email)
Genre(GenreId, Name)
Invoice(InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.

## Letting the magic happen! Creating agent with a stronger model ... 

In [19]:
agent = create_agent(
    model=llm_noreason,
    tools=[sql_query]
)

messages = [
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response = agent.invoke({"messages": messages})
pp(f"Model used: {llm_noreason.model_name}")
pp(f"Response: {response['messages'][-1].content}")

'Model used: databricks-qwen35-122b-a10b'
"Response: The most popular artist beginning with 'S' in this database is **Smashing Pumpkins**, with 34 tracks."


In [20]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Who is the most popular artist beginning with 'S' in this database?
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_79389285-9722-470e-a3d6-51bb29dde54f)
 Call ID: call_79389285-9722-470e-a3d6-51bb29dde54f
  Args:
    query: SELECT artist_name, COUNT(*) as song_count FROM artists WHERE artist_name LIKE 'S%' GROUP BY artist_name ORDER BY song_count DESC LIMIT 1;
================================= Tool Message =================================
Name: sql_query

Error: (sqlite3.OperationalError) no such table: artists
[SQL: SELECT artist_name, COUNT(*) as song_count FROM artists WHERE artist_name LIKE 'S%' GROUP BY artist_name ORDER BY song_count DESC LIMIT 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_bcce15ac-5a1f-43f8